# RAG Pipeline with LangChain and a 100-Page PDF

This notebook teaches a complete Retrieval-Augmented Generation, or RAG, pipeline using LangChain.

We will use a real long PDF from the internet and process the first 100 pages for classroom speed.

PDF used in this lesson:

- Book: *The Elements of Statistical Learning* by Trevor Hastie, Robert Tibshirani, and Jerome Friedman
- Official book page: https://hastie.su.domains/ElemStatLearn/
- Direct PDF URL used for download in this notebook: https://esl.hohoweiya.xyz/book/The%20Elements%20of%20Statistical%20Learning.pdf

What students will learn:

1. What RAG is
2. Downloading a PDF from the internet
3. Loading PDF pages with LangChain
4. Selecting the first 100 pages
5. Splitting text into chunks
6. Creating embeddings
7. Storing embeddings in FAISS vector database
8. Retrieving relevant chunks
9. Building a prompt from retrieved context
10. Generating answers with an LLM
11. Showing source pages
12. Saving and loading the vector database
13. Testing and improving a RAG system

## 1. What Is RAG?

RAG means Retrieval-Augmented Generation.

A normal LLM answers from what it already learned during training. A RAG system first retrieves relevant information from your documents, then asks the LLM to answer using that retrieved context.

Simple RAG flow:

1. Load documents.
2. Split documents into smaller chunks.
3. Convert chunks into embeddings.
4. Store embeddings in a vector database.
5. Retrieve chunks similar to the user question.
6. Send retrieved context plus question to an LLM.
7. Return an answer with sources.

Why RAG is useful:

- It can answer from private or custom documents.
- It reduces hallucination when the prompt requires evidence.
- It can show source pages.
- It can be updated by changing documents instead of retraining a model.

## 2. Install Required Packages

This cell installs LangChain, PDF loading tools, FAISS, and embedding tools.

The first run may take a few minutes because `sentence-transformers` downloads a small embedding model.

In [ ]:
# If a package is missing, this cell installs it into the current notebook kernel.
import importlib.util
import subprocess
import sys

required_packages = {
    "langchain": "langchain",
    "langchain_community": "langchain-community",
    "langchain_text_splitters": "langchain-text-splitters",
    "langchain_openai": "langchain-openai",
    "langchain_huggingface": "langchain-huggingface",
    "pypdf": "pypdf",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "requests": "requests",
    "dotenv": "python-dotenv",
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    try:
        from IPython import get_ipython

        ipython = get_ipython()
        if ipython is not None:
            ipython.run_line_magic("pip", "install " + " ".join(missing_packages))
        else:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    except Exception as error:
        print("Automatic installation did not work in this Python environment.")
        print("Please run this in a notebook cell instead:")
        print("%pip install langchain langchain-community langchain-text-splitters langchain-openai langchain-huggingface pypdf faiss-cpu sentence-transformers requests python-dotenv")
        raise error
else:
    print("All required packages are already installed.")

## 3. Imports and Settings

In [ ]:
import os
from pathlib import Path
import textwrap
import requests

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

PDF_URL = "https://esl.hohoweiya.xyz/book/The%20Elements%20of%20Statistical%20Learning.pdf"
PDF_FILE = Path("elements_of_statistical_learning.pdf")
PAGE_LIMIT = 100
FAISS_INDEX_DIR = "esl_100_pages_faiss_index"

print("RAG notebook is ready.")

## 4. Download the PDF

This cell downloads the PDF only if it is not already present.

In [ ]:
def download_file(url, output_path):
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"File already exists: {output_path} ({output_path.stat().st_size / 1024 / 1024:.2f} MB)")
        return output_path

    print("Downloading PDF. This may take a moment...")
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with output_path.open("wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

    print(f"Downloaded: {output_path} ({output_path.stat().st_size / 1024 / 1024:.2f} MB)")
    return output_path

pdf_path = download_file(PDF_URL, PDF_FILE)

## 5. Load the First 100 Pages with LangChain

`PyPDFLoader` loads each PDF page as a LangChain `Document` object.

Each `Document` has:

- `page_content`: text from the page
- `metadata`: information such as source file and page number

In [ ]:
loader = PyPDFLoader(str(pdf_path))

# lazy_load reads pages one by one. We stop at PAGE_LIMIT for speed.
pages = []
for page_number, document in enumerate(loader.lazy_load(), start=1):
    if page_number > PAGE_LIMIT:
        break
    document.metadata["page_number"] = page_number
    document.metadata["source_title"] = "The Elements of Statistical Learning"
    pages.append(document)

print("Loaded pages:", len(pages))
print("First page metadata:", pages[0].metadata)

In [ ]:
# Preview text from the first loaded page.
preview = pages[0].page_content[:1000]
print(textwrap.fill(preview, width=100))

## 6. Clean and Inspect Page Text

PDF text extraction is not always perfect. It can include headers, footers, broken lines, and page artifacts.

For a simple first RAG system, we will keep the extracted text as-is. In production, you may add more PDF-specific cleaning.

In [ ]:
page_lengths = [len(page.page_content) for page in pages]

print("Shortest page length:", min(page_lengths))
print("Longest page length:", max(page_lengths))
print("Average page length:", round(sum(page_lengths) / len(page_lengths), 1))

## 7. Split Pages into Chunks

LLMs and embedding models work better with chunks than with huge documents.

Chunking settings:

- `chunk_size`: approximate number of characters per chunk
- `chunk_overlap`: repeated characters between chunks so context is not cut too sharply

Good chunk sizes depend on the document and model. For many PDF RAG systems, 800 to 1500 characters is a reasonable starting point.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(pages)

print("Original pages:", len(pages))
print("Text chunks:", len(chunks))
print("First chunk metadata:", chunks[0].metadata)
print("First chunk preview:")
print(textwrap.fill(chunks[0].page_content[:800], width=100))

## 8. Create Embeddings

An embedding converts text into a vector of numbers that represents meaning.

In this notebook we use a local open-source embedding model:

`sentence-transformers/all-MiniLM-L6-v2`

This avoids needing an API key for embeddings.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Test one embedding so students can see the vector size.
test_vector = embedding_model.embed_query("What is statistical learning?")
print("Embedding vector length:", len(test_vector))
print("First 5 numbers:", [round(value, 4) for value in test_vector[:5]])

## 9. Store Embeddings in FAISS

FAISS is a vector database library. It lets us search for chunks that are semantically similar to a question.

This step creates embeddings for all chunks and stores them in a FAISS index.

In [ ]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
print("FAISS vectorstore created.")
print("Number of indexed chunks:", vectorstore.index.ntotal)

## 10. Run a Similarity Search

Before adding an LLM, always test retrieval.

If retrieval is poor, generation will also be poor.

In [ ]:
query = "What is the difference between supervised and unsupervised learning?"

retrieved_docs = vectorstore.similarity_search(query, k=4)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved chunk {i} ---")
    print("Page:", doc.metadata.get("page_number"))
    print(textwrap.fill(doc.page_content[:600], width=100))

## 11. Create a Retriever

A retriever is a standard LangChain interface for fetching relevant documents.

We will use it inside the RAG chain.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

docs_for_question = retriever.invoke(query)
print("Retrieved documents:", len(docs_for_question))

## 12. Build the RAG Prompt

A good RAG prompt tells the LLM to answer only from the context and to admit when the answer is not available.

This reduces hallucination.

In [ ]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful teaching assistant for data science students.
Answer the question using only the context below.
If the context does not contain the answer, say that the answer is not available in the provided PDF pages.
Keep the answer clear and beginner-friendly.

Context:
{context}

Question: {question}

Answer:
    """
)

def format_docs(docs):
    formatted_chunks = []
    for doc in docs:
        page = doc.metadata.get("page_number", "unknown")
        formatted_chunks.append(f"[Page {page}]\n{doc.page_content}")
    return "\n\n".join(formatted_chunks)

print("Prompt template ready.")

## 13. No-Key Classroom Fallback: Build the Prompt Manually

If students do not have an LLM API key yet, they can still learn how RAG prepares the context.

This cell retrieves the relevant chunks and builds the exact prompt that would be sent to the LLM.

In [ ]:
def build_rag_prompt(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    messages = prompt.format_messages(context=context, question=question)
    return messages[0].content, docs

manual_prompt, source_docs = build_rag_prompt("What is statistical learning?")

print(manual_prompt[:2000])
print("\nSource pages:", sorted({doc.metadata.get("page_number") for doc in source_docs}))

## 14. Generation with an LLM

To generate final natural-language answers, set an OpenAI API key before running this cell.

In a notebook cell, you can set it like this:

```python
import os
os.environ["OPENAI_API_KEY"] = "your_api_key_here"
```

You can also set `OPENAI_MODEL`. If you do not set it, the notebook uses `gpt-4o-mini` as the default example model.

In [ ]:
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print("LLM RAG chain is ready with model:", OPENAI_MODEL)
else:
    rag_chain = None
    print("OPENAI_API_KEY is not set. Use the no-key retrieval and prompt cells, or set your key and rerun this cell.")

## 15. Ask Questions with the RAG Chain

In [ ]:
def ask_rag(question):
    source_docs = retriever.invoke(question)
    source_pages = sorted({doc.metadata.get("page_number") for doc in source_docs})

    if rag_chain is None:
        manual_prompt, _ = build_rag_prompt(question)
        return {
            "answer": "LLM generation is not available because OPENAI_API_KEY is not set. The retrieved context is shown in the prompt preview.",
            "source_pages": source_pages,
            "prompt_preview": manual_prompt[:2000],
        }

    answer = rag_chain.invoke(question)
    return {
        "answer": answer,
        "source_pages": source_pages,
    }

result = ask_rag("What is statistical learning, explained simply?")
print("Answer:\n", result["answer"])
print("\nSource pages:", result["source_pages"])

if "prompt_preview" in result:
    print("\nPrompt preview:\n")
    print(result["prompt_preview"])

## 16. Ask Multiple Questions

In [ ]:
questions = [
    "What are supervised learning and unsupervised learning?",
    "What is linear regression used for?",
    "What does model assessment mean?",
]

for question in questions:
    print("\n" + "=" * 100)
    print("Question:", question)
    result = ask_rag(question)
    print("Source pages:", result["source_pages"])
    print("Answer:", result["answer"][:1200])

## 17. Show Sources Clearly

A good RAG application should show where the answer came from.

This helps users trust and verify the answer.

In [ ]:
def show_sources(question, k=4):
    docs = vectorstore.similarity_search(question, k=k)
    for i, doc in enumerate(docs, start=1):
        print(f"\nSource {i}")
        print("Page:", doc.metadata.get("page_number"))
        print("Text preview:")
        print(textwrap.fill(doc.page_content[:700], width=100))

show_sources("Explain bias and variance in model fitting.")

## 18. Save and Reload the Vector Database

Creating embeddings can take time. Save the FAISS index so you can reuse it later.

Important safety note: `allow_dangerous_deserialization=True` should only be used for indexes you created yourself or fully trust.

In [ ]:
vectorstore.save_local(FAISS_INDEX_DIR)
print("Saved FAISS index to:", FAISS_INDEX_DIR)

reloaded_vectorstore = FAISS.load_local(
    FAISS_INDEX_DIR,
    embedding_model,
    allow_dangerous_deserialization=True,
)

print("Reloaded chunks:", reloaded_vectorstore.index.ntotal)

## 19. Retrieval Parameters to Experiment With

Important parameters:

- `k`: number of chunks retrieved
- `chunk_size`: chunk length before embedding
- `chunk_overlap`: repeated text between neighboring chunks
- `search_type`: similarity or Maximal Marginal Relevance, also called MMR

MMR can improve diversity when similar chunks repeat the same idea.

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 12},
)

mmr_docs = mmr_retriever.invoke("What is model complexity?")

for i, doc in enumerate(mmr_docs, start=1):
    print(f"Result {i}: page {doc.metadata.get('page_number')}")

## 20. Simple RAG Evaluation

RAG systems should be tested.

Basic classroom checks:

- Did retrieval return relevant pages?
- Did the answer use only the retrieved context?
- Did the answer mention uncertainty when context was missing?
- Are sources shown?
- Is the answer understandable to the target audience?

In [ ]:
evaluation_questions = [
    "What is supervised learning?",
    "What is unsupervised learning?",
    "What is the purpose of training data?",
    "What is the capital city of India?",
]

for question in evaluation_questions:
    docs = retriever.invoke(question)
    pages = sorted({doc.metadata.get("page_number") for doc in docs})
    print("Question:", question)
    print("Retrieved pages:", pages)
    print("Top chunk preview:", docs[0].page_content[:250].replace("\n", " "))
    print("-" * 100)

## 21. Common RAG Problems and Fixes

| Problem | Possible Fix |
|---|---|
| Bad PDF text extraction | Try another parser, OCR, or document cleaning |
| Retrieved chunks are irrelevant | Change chunk size, embedding model, or retrieval method |
| Answer is too vague | Improve prompt or retrieve more chunks |
| Answer hallucinates | Tell the model to answer only from context and show sources |
| Too slow | Use fewer pages, smaller embedding model, cached vector database |
| Too expensive | Use local embeddings and cache results |
| Missing answer | Add more documents or increase page limit |

In production, always log questions, retrieved chunks, answers, and user feedback.

## 22. Student Exercises

1. Change `PAGE_LIMIT` from 100 to 50 and compare the number of chunks.
2. Change `chunk_size` from 1000 to 500. What happens to the number of chunks?
3. Change retriever `k` from 4 to 8. Does the answer improve?
4. Compare similarity search with MMR search.
5. Ask a question that is not answered in the first 100 pages. Does the prompt handle it properly?
6. Save the retrieved chunks for each question in a CSV file.
7. Try a different PDF with more than 100 pages.
8. Add a simple user interface with Streamlit or Gradio.
9. Create a small evaluation table with question, expected topic, retrieved pages, and answer quality.
10. Explain why RAG is different from fine-tuning.

## 23. Final Checklist for a RAG Project

Before calling a RAG system complete, check:

1. The source documents are correct and allowed to be used.
2. Text extraction quality is acceptable.
3. Chunks are not too small or too large.
4. Embeddings are created successfully.
5. Vector search retrieves relevant context.
6. The prompt tells the model to use only context.
7. Answers include source pages.
8. The system handles unknown questions honestly.
9. The vector database is saved and reusable.
10. Evaluation questions are tested before showing the system to users.